NOTEBOOK : `05_pipeline_orchestrator`

PURPOSE  : Calls NB02 → NB03 → NB04 in sequence
       One cell per notebook call
       
RUN      : Scheduled via Databricks Job every 6 hours

# 05 · Pipeline orchestrator
Runs the full Customer 360 pipeline by calling each
notebook in sequence using `dbutils.notebook.run()`.

| Cell | Notebook called | Stage |
|------|----------------|-------|
| 3 | 02_incremental_generator | Append Bronze rows |
| 4 | 03_silver_transform | Rebuild Silver |
| 5 | 04_gold_metrics | Rebuild Gold |

**NB01 is excluded — it runs once manually only.**



## Cell 1 — Config
Update `NOTEBOOK_BASE` to your workspace path before running.
Find it by right-clicking any notebook → Copy path → strip the filename.


In [0]:

from datetime import datetime

PIPELINE_START = datetime.utcnow()
PIPELINE_ID    = "PIPELINE_" + PIPELINE_START.strftime("%Y%m%d_%H%M%S")

# ── UPDATE THIS PATH ──────────────────────────────────────────
NOTEBOOK_BASE = "/Workspace/Users/sskws1234@gmail.com/Customer360/notebooks"
# ─────────────────────────────────────────────────────────────

TIMEOUT = 1800   # 30 minutes max per notebook

print(f"Pipeline ID : {PIPELINE_ID}")
print(f"Start UTC   : {PIPELINE_START.strftime('%Y-%m-%d %H:%M:%S')}")
print(f"Base path   : {NOTEBOOK_BASE}")

## Cell 2 — Verify notebook paths exist before running

In [0]:

import os

notebooks = [
    "02_incremental_generator",
    "03_silver_transform",
    "04_gold_metrics",
]

for nb in notebooks:
    path = f"{NOTEBOOK_BASE}/{nb}"
    print(f"  Will call : {path}")

print("\nUpdate NOTEBOOK_BASE in Cell 1 if any path looks wrong.")

## Cell 3 — Run NB02 · Incremental generator

In [0]:
print(f"[{datetime.utcnow().strftime('%H:%M:%S')}] Starting 02_incremental_generator ...")

run_id_02 = dbutils.notebook.run(
    f"{NOTEBOOK_BASE}/02_incremental_generator",
    timeout_seconds=TIMEOUT,
    arguments={}
)

print(f"[{datetime.utcnow().strftime('%H:%M:%S')}] 02_incremental_generator complete")
print(f"  Returned : {run_id_02}")

## Cell 4 — Run NB03 · Silver transform

In [0]:

print(f"[{datetime.utcnow().strftime('%H:%M:%S')}] Starting 03_silver_transform ...")

run_id_03 = dbutils.notebook.run(
    f"{NOTEBOOK_BASE}/03_silver_transform",
    timeout_seconds=TIMEOUT,
    arguments={}
)

print(f"[{datetime.utcnow().strftime('%H:%M:%S')}] 03_silver_transform complete")
print(f"  Returned : {run_id_03}")

## Cell 5 — Run NB04 · Gold metrics

In [0]:

print(f"[{datetime.utcnow().strftime('%H:%M:%S')}] Starting 04_gold_metrics ...")

run_id_04 = dbutils.notebook.run(
    f"{NOTEBOOK_BASE}/04_gold_metrics",
    timeout_seconds=TIMEOUT,
    arguments={}
)

print(f"[{datetime.utcnow().strftime('%H:%M:%S')}] 04_gold_metrics complete")
print(f"  Returned : {run_id_04}")

## Cell 6 — Pipeline summary

In [0]:

pipeline_end  = datetime.utcnow()
duration_secs = int((pipeline_end - PIPELINE_START).total_seconds())

print("=" * 55)
print(f"  PIPELINE COMPLETE")
print(f"  Pipeline ID : {PIPELINE_ID}")
print(f"  Duration    : {duration_secs//60}m {duration_secs%60}s")
print("─" * 55)
print(f"  NB02 run ID : {run_id_02}")
print(f"  NB03 run ID : {run_id_03}")
print(f"  NB04 run ID : {run_id_04}")
print("=" * 55)

dbutils.notebook.exit(PIPELINE_ID)